# ETL Driblab Capstone

In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')

print('Setup complete ✓')

Setup complete ✓


In [2]:
rawevents = json.load(open("/Users/nataliaurrea/Documents/IE/MBDS/Term III/Driblab/data/raw/683231_events.json"))

df = pd.json_normalize(rawevents)

In [3]:
df.rename(columns={
    'event.id':              'event_id',
    'event.event_type_id':   'event_type_id',
    'event.event_type_name': 'event_type_name',
    'team.team_id':          'team_id',
    'team.team_name':        'team_name',
    'player.player_id':      'player_id',
    'player.player_name':    'player_name',
}, inplace=True)
df.head()

,match_id,period_id,min,sec,x,y,outcome,qualifiers,possession_id,xa,xg,xt,x_start,y_start,x_end,y_end,milisec,event_id,event_type_id,event_type_name,team_id,team_name,player_id,player_name
0,683231,1,0,8,51.0,50.0,True,"[{'qualifier_id': 279, 'qualifier_name': 'Kick...",2.0,0.0,0.0,0.0,51.0,50.0,29.0,33.0,855,561575035,1,PASS,4279,FC Barcelona,1348155,Fermin Lopez
1,683231,1,0,11,29.2,33.0,False,"[{'qualifier_id': 74, 'qualifier_name': 'High'...",2.0,0.0,0.0,0.0,29.0,33.0,NaN,NaN,668,561575036,1,PASS,4279,FC Barcelona,1303293,Ronald Araujo
2,683231,1,0,15,22.7,39.3,True,"[{'qualifier_id': 3, 'qualifier_name': 'Head',...",3.0,0.0,0.0,0.0,23.0,39.0,36.0,22.0,753,561575037,1,PASS,4263,Celta de Vigo,1293412,Carl Starfelt
3,683231,1,0,17,36.2,21.7,False,"[{'qualifier_id': 140, 'qualifier_name': 'Pass...",3.0,0.0,0.0,0.0,36.0,22.0,NaN,NaN,725,561575038,1,PASS,4263,Celta de Vigo,1331661,Ferran Jutgla
4,683231,1,0,18,60.5,68.8,True,"[{'qualifier_id': None, 'qualifier_name': None...",4.0,0.0,0.0,0.0,60.0,69.0,60.0,69.0,380,46457652,49,BALL RECOVERY,4279,FC Barcelona,162233,Frenkie De Jong


In [4]:
dim = pd.read_csv("/Users/nataliaurrea/Documents/IE/MBDS/Term III/Driblab/data/raw/dim_event_type.csv")
dim.head()
dim.tail()


,event_id,category_id,name_eng,name_esp,des_eng,des_esp
79,121,2,YELLOW CARD,TARJETA AMARILLA,NaN,NaN
80,123,2,RED CARD,TARJETA ROJA,NaN,NaN
81,124,2,GOAL KICK,SAQUE DE PUERTA,NaN,NaN
82,125,4,TACKLE WITH POSSESSION,TACKLE CON POSESIÓN,NaN,NaN
83,126,4,TACKLE WITHOUT POSSESSION,TACKLE SIN POSESIÓN,NaN,NaN


In [5]:
dim.rename(columns={'event_id': 'event_type_id'}, inplace=True)

In [6]:
df = df.merge(
    dim[[ 'event_type_id', 'category_id', 'name_eng', 'des_eng']],
    on='event_type_id',
    how='left'
)
df.head()

,match_id,period_id,min,sec,x,y,outcome,qualifiers,possession_id,xa,xg,xt,x_start,y_start,x_end,y_end,milisec,event_id,event_type_id,event_type_name,team_id,team_name,player_id,player_name,category_id,name_eng,des_eng
0,683231,1,0,8,51.0,50.0,True,"[{'qualifier_id': 279, 'qualifier_name': 'Kick...",2.0,0.0,0.0,0.0,51.0,50.0,29.0,33.0,855,561575035,1,PASS,4279,FC Barcelona,1348155,Fermin Lopez,2,PASS,Any pass attempted from one player to another ...
1,683231,1,0,11,29.2,33.0,False,"[{'qualifier_id': 74, 'qualifier_name': 'High'...",2.0,0.0,0.0,0.0,29.0,33.0,NaN,NaN,668,561575036,1,PASS,4279,FC Barcelona,1303293,Ronald Araujo,2,PASS,Any pass attempted from one player to another ...
2,683231,1,0,15,22.7,39.3,True,"[{'qualifier_id': 3, 'qualifier_name': 'Head',...",3.0,0.0,0.0,0.0,23.0,39.0,36.0,22.0,753,561575037,1,PASS,4263,Celta de Vigo,1293412,Carl Starfelt,2,PASS,Any pass attempted from one player to another ...
3,683231,1,0,17,36.2,21.7,False,"[{'qualifier_id': 140, 'qualifier_name': 'Pass...",3.0,0.0,0.0,0.0,36.0,22.0,NaN,NaN,725,561575038,1,PASS,4263,Celta de Vigo,1331661,Ferran Jutgla,2,PASS,Any pass attempted from one player to another ...
4,683231,1,0,18,60.5,68.8,True,"[{'qualifier_id': None, 'qualifier_name': None...",4.0,0.0,0.0,0.0,60.0,69.0,60.0,69.0,380,46457652,49,BALL RECOVERY,4279,FC Barcelona,162233,Frenkie De Jong,4,BALL RECOVERY,When a player takes possession of a loose ball


In [7]:
print(df.shape)

(1752, 27)


In [8]:
print(df.columns)

Index(['match_id', 'period_id', 'min', 'sec', 'x', 'y', 'outcome',
       'qualifiers', 'possession_id', 'xa', 'xg', 'xt', 'x_start', 'y_start',
       'x_end', 'y_end', 'milisec', 'event_id', 'event_type_id',
       'event_type_name', 'team_id', 'team_name', 'player_id', 'player_name',
       'category_id', 'name_eng', 'des_eng'],
      dtype='str')


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1752 entries, 0 to 1751
Data columns (total 27 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   match_id         1752 non-null   int64  
 1   period_id        1752 non-null   int64  
 2   min              1752 non-null   int64  
 3   sec              1752 non-null   int64  
 4   x                1752 non-null   float64
 5   y                1752 non-null   float64
 6   outcome          1752 non-null   bool   
 7   qualifiers       1752 non-null   object 
 8   possession_id    1694 non-null   float64
 9   xa               1694 non-null   float64
 10  xg               1694 non-null   float64
 11  xt               1694 non-null   float64
 12  x_start          1419 non-null   float64
 13  y_start          1419 non-null   float64
 14  x_end            1189 non-null   float64
 15  y_end            1189 non-null   float64
 16  milisec          1752 non-null   int64  
 17  event_id         1752 non

In [10]:
df.isnull().sum() 

match_id             0
period_id            0
min                  0
sec                  0
x                    0
y                    0
outcome              0
qualifiers           0
possession_id       58
xa                  58
xg                  58
xt                  58
x_start            333
y_start            333
x_end              563
y_end              563
milisec              0
event_id             0
event_type_id        0
event_type_name      0
team_id              0
team_name            0
player_id            0
player_name          0
category_id          0
name_eng             0
des_eng              0
dtype: int64

In [11]:
df.drop(columns=['qualifiers']).duplicated().sum() #drop qualifiers as it has a dictionary

np.int64(0)

In [12]:
df['event_type_name'].value_counts()

event_type_name
PASS                1217
BALL TOUCH           148
TACKLE                66
TAKEON                53
FOUL                  45
BALL RECOVERY         41
INTERCEPTION          32
AERIAL                30
CHALLENGE             23
SAVED SHOT            15
SAVE                  13
DISPOSSESSED          13
CLEARANCE             10
SUBSTITUTION OFF       9
SUBSTITUTION ON        9
OFFSIDE PASS           7
GOAL                   6
CARD                   6
MISSED SHOT            3
SHOT ON POST           2
END                    2
CLAIM                  1
FORMATION CHANGE       1
Name: count, dtype: int64

In [13]:
team_events = df.groupby(['team_name','event_type_name']).size().unstack(fill_value=0)
team_events.head()

event_type_name,AERIAL,BALL RECOVERY,BALL TOUCH,CARD,CHALLENGE,CLAIM,CLEARANCE,DISPOSSESSED,END,FORMATION CHANGE,FOUL,GOAL,INTERCEPTION,MISSED SHOT,OFFSIDE PASS,PASS,SAVE,SAVED SHOT,SHOT ON POST,SUBSTITUTION OFF,SUBSTITUTION ON,TACKLE,TAKEON
team_name,,,,,,,,,,,,,,,,,,,,,,,
Celta de Vigo,15,20,68,1,19,1,6,9,0,0,23,2,19,1,5,475,11,2,0,5,5,29,15
FC Barcelona,15,21,80,5,4,0,4,4,0,1,22,4,13,2,2,742,2,13,2,4,4,37,38
Retired,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0


FC Barcelona generated more passes, leading to mayority ball possesion

In [14]:
team_events_pct = team_events.div(team_events.sum(axis=1), axis=0).mul(100).round(1)
team_events_pct.head()

event_type_name,AERIAL,BALL RECOVERY,BALL TOUCH,CARD,CHALLENGE,CLAIM,CLEARANCE,DISPOSSESSED,END,FORMATION CHANGE,FOUL,GOAL,INTERCEPTION,MISSED SHOT,OFFSIDE PASS,PASS,SAVE,SAVED SHOT,SHOT ON POST,SUBSTITUTION OFF,SUBSTITUTION ON,TACKLE,TAKEON
team_name,,,,,,,,,,,,,,,,,,,,,,,
Celta de Vigo,2.1,2.7,9.3,0.1,2.6,0.1,0.8,1.2,0.0,0.0,3.1,0.3,2.6,0.1,0.7,65.0,1.5,0.3,0.0,0.7,0.7,4.0,2.1
FC Barcelona,1.5,2.1,7.9,0.5,0.4,0.0,0.4,0.4,0.0,0.1,2.2,0.4,1.3,0.2,0.2,72.8,0.2,1.3,0.2,0.4,0.4,3.6,3.7
Retired,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
top_10_events = df['event_type_name'].value_counts().head(10).index
team_events_pct[top_10_events]

event_type_name,PASS,BALL TOUCH,TACKLE,TAKEON,FOUL,BALL RECOVERY,INTERCEPTION,AERIAL,CHALLENGE,SAVED SHOT
team_name,,,,,,,,,,
Celta de Vigo,65.0,9.3,4.0,2.1,3.1,2.7,2.6,2.1,2.6,0.3
FC Barcelona,72.8,7.9,3.6,3.7,2.2,2.1,1.3,1.5,0.4,1.3
Retired,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
df.groupby('event_type_name')['outcome'].mean().mul(100).round(1).sort_values(ascending=False)

event_type_name
GOAL                100.0
END                 100.0
SAVE                100.0
SUBSTITUTION OFF    100.0
INTERCEPTION        100.0
BALL RECOVERY       100.0
SHOT ON POST        100.0
FORMATION CHANGE    100.0
DISPOSSESSED        100.0
CLEARANCE           100.0
SUBSTITUTION ON     100.0
CARD                100.0
BALL TOUCH          100.0
SAVED SHOT          100.0
PASS                 88.2
MISSED SHOT          66.7
TACKLE               50.0
FOUL                 48.9
TAKEON               47.2
AERIAL               40.0
OFFSIDE PASS          0.0
CLAIM                 0.0
CHALLENGE             0.0
Name: outcome, dtype: float64

## Raw tracking data columns and nested position fields

This section inspects the raw `*_tracking_data.jsonl` files. Each tracking file has a metadata header on the first line, followed by one JSON object per frame.

The goal is to see every tracking frame column and unpack the nested fields that carry pitch positions:

- `ball`: ball x/y/z coordinates
- `data`: team and player tracking rows, including player x/y positions and visibility
- `cam`: camera/calibration points when available


In [17]:
import json
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
tracking_files = sorted(RAW_DATA_DIR.glob("*_tracking_data.jsonl"))

print(f"Tracking files found: {len(tracking_files)}")
print([path.name for path in tracking_files[:10]])

# Use a match with populated player tracking data for readable nested examples.
preferred_tracking_path = RAW_DATA_DIR / "679026_tracking_data.jsonl"
tracking_path = preferred_tracking_path if preferred_tracking_path.exists() else tracking_files[0]
match_id = tracking_path.name.split("_", 1)[0]

with tracking_path.open() as f:
    tracking_header = json.loads(f.readline())
    tracking_frames = [json.loads(line) for line in f]

print(f"Example tracking file: {tracking_path.name}")
print(f"Frames loaded: {len(tracking_frames):,}")
print(f"Header keys: {list(tracking_header.keys())}")
print(f"FPS: {tracking_header.get('FPS')}")


Tracking files found: 33
['678949_tracking_data.jsonl', '679026_tracking_data.jsonl', '679053_tracking_data.jsonl', '679072_tracking_data.jsonl', '679075_tracking_data.jsonl', '679088_tracking_data.jsonl', '679104_tracking_data.jsonl', '679128_tracking_data.jsonl', '682607_tracking_data.jsonl', '682717_tracking_data.jsonl']
Example tracking file: 679026_tracking_data.jsonl
Frames loaded: 59,502
Header keys: ['match_data', 'teams_data', 'players_data', 'FPS']
FPS: 10


In [18]:
# Collect every raw tracking frame column seen across all tracking files.
tracking_column_rows = []
for path in tracking_files:
    this_match_id = path.name.split("_", 1)[0]
    with path.open() as f:
        header = json.loads(f.readline())
        for row_number, line in enumerate(f):
            frame = json.loads(line)
            for column, value in frame.items():
                if isinstance(value, dict):
                    value_type = "dict"
                    nested_size = len(value)
                elif isinstance(value, list):
                    value_type = "list"
                    nested_size = len(value)
                elif value is None:
                    value_type = "null"
                    nested_size = None
                else:
                    value_type = type(value).__name__
                    nested_size = None
                tracking_column_rows.append({
                    "match_id": this_match_id,
                    "column": column,
                    "example_type": value_type,
                    "example_nested_size": nested_size,
                    "example_value": value,
                })
            break

tracking_column_examples = pd.DataFrame(tracking_column_rows)
tracking_column_summary = (
    tracking_column_examples
    .groupby("column", as_index=False)
    .agg(
        matches_seen=("match_id", "nunique"),
        example_type=("example_type", "first"),
        example_nested_size=("example_nested_size", "first"),
        example_value=("example_value", "first"),
    )
)

# Keep nested examples readable in the table.
def compact_json(value, max_chars=220):
    if isinstance(value, (dict, list)):
        text = json.dumps(value, ensure_ascii=False)
    else:
        text = str(value)
    return text if len(text) <= max_chars else text[:max_chars] + "..."

tracking_column_summary["example_value"] = tracking_column_summary["example_value"].apply(compact_json)
display(tracking_column_summary)


,column,matches_seen,example_type,example_nested_size,example_value
0,Videotimestamp,33,float,NaN,1.0
1,ball,33,list,3.0,"[null, null, null]"
2,cam,33,null,4.0,"[[11.0, -4.44], [3059.81, -2795.66], [43.6, 79..."
3,data,33,dict,0.0,{}
4,frame,33,int,NaN,0
5,match_clock,33,list,2.0,"[0, 1]"
6,period,33,int,NaN,1


In [19]:
# Raw frame table for the example match.
# This keeps the original tracking columns exactly as they appear in JSONL.
tracking_frame_table = pd.DataFrame(tracking_frames)

print(f"Raw tracking frame table shape for {match_id}: {tracking_frame_table.shape}")
display(tracking_frame_table.head(10))


Raw tracking frame table shape for 679026: (59502, 7)


,data,match_clock,frame,Videotimestamp,period,ball,cam
0,"{'8561': [{'x': 48.88, 'y': 61.48, 'vis': Fals...","[0, 1]",0,1.0,1,"[None, None, None]","[[11.0, -4.44], [3059.81, -2795.66], [43.6, 79..."
1,"{'8561': [{'x': 50.46, 'y': 61.94, 'vis': Fals...","[0, 1]",1,1.1,1,"[None, None, None]","[[11.0, -4.59], [3042.07, -2788.7], [43.6, 80...."
2,"{'8561': [{'x': 52.03, 'y': 62.39, 'vis': Fals...","[0, 1]",2,1.2,1,"[47.21, 47.87, 0.925]","[[11.0, -4.74], [3024.32, -2781.74], [43.59, 8..."
3,"{'8561': [{'x': 53.61, 'y': 62.85, 'vis': Fals...","[0, 1]",3,1.3,1,"[46.98, 48.22, 0.556]","[[996.69, 2492.21], [-177.63, 223.3], [43.72, ..."
4,"{'8561': [{'x': 55.19, 'y': 63.31, 'vis': Fals...","[0, 1]",4,1.4,1,"[46.74, 48.58, 0.187]","[[1982.38, 4989.16], [-3379.58, 3228.33], [43...."
5,"{'8561': [{'x': 54.5, 'y': 63.5, 'vis': False,...","[0, 1]",5,1.5,1,"[49.99, 41.22, 0.14]","[[1271.4, 3166.92], [-2642.14, 2554.49], [44.0..."
6,"{'8561': [{'x': 53.82, 'y': 63.69, 'vis': Fals...","[0, 1]",6,1.6,1,"[53.24, 33.85, 0.094]","[[560.43, 1344.68], [-1904.7, 1880.64], [44.18..."
7,"{'8561': [{'x': 53.13, 'y': 63.88, 'vis': True...","[0, 1]",7,1.7,1,"[50.78, 42.0, 0.047]","[[285.72, 671.33], [-621.52, 661.94], [43.82, ..."
8,"{'8561': [{'x': 52.44, 'y': 64.08, 'vis': True...","[0, 1]",8,1.8,1,"[48.33, 50.15, 0.0]","[[11.0, -2.02], [661.66, -556.77], [43.47, 80...."
9,"{'8561': [{'x': 51.76, 'y': 64.27, 'vis': True...","[0, 1]",9,1.9,1,"[48.37, 49.22, 0.209]","[[11.0, -2.44], [1798.24, -1625.5], [43.56, 80..."


### Unpacked nested tracking fields

The raw frame table above shows nested values as lists/dicts. The next cells unpack those nested structures into readable tables.

In [20]:
# Ball coordinates: [x, y, z]
ball_rows = []
for frame in tracking_frames[:200]:
    ball = frame.get("ball") or [None, None, None]
    clock = frame.get("match_clock") or [None, None]
    ball_rows.append({
        "match_id": match_id,
        "period": frame.get("period"),
        "frame": frame.get("frame"),
        "Videotimestamp": frame.get("Videotimestamp"),
        "match_clock_min": clock[0] if len(clock) > 0 else None,
        "match_clock_sec": clock[1] if len(clock) > 1 else None,
        "ball_x": ball[0] if len(ball) > 0 else None,
        "ball_y": ball[1] if len(ball) > 1 else None,
        "ball_z": ball[2] if len(ball) > 2 else None,
    })

ball_positions = pd.DataFrame(ball_rows)
display(ball_positions.head(30))


,match_id,period,frame,Videotimestamp,match_clock_min,match_clock_sec,ball_x,ball_y,ball_z
0,679026,1,0,1.0,0,1,NaN,NaN,NaN
1,679026,1,1,1.1,0,1,NaN,NaN,NaN
2,679026,1,2,1.2,0,1,47.21,47.87,0.925
3,679026,1,3,1.3,0,1,46.98,48.22,0.556
4,679026,1,4,1.4,0,1,46.74,48.58,0.187
5,679026,1,5,1.5,0,1,49.99,41.22,0.140
6,679026,1,6,1.6,0,1,53.24,33.85,0.094
7,679026,1,7,1.7,0,1,50.78,42.00,0.047
8,679026,1,8,1.8,0,1,48.33,50.15,0.000
9,679026,1,9,1.9,0,1,48.37,49.22,0.209


In [21]:
# Player positions inside the nested `data` column.
# Structure: data -> team_id -> list of players with x/y/vis/id.
player_rows = []
for frame in tracking_frames:
    teams = frame.get("data") or {}
    if not teams:
        continue
    clock = frame.get("match_clock") or [None, None]
    for team_id, players in teams.items():
        for player in players:
            player_rows.append({
                "match_id": match_id,
                "period": frame.get("period"),
                "frame": frame.get("frame"),
                "Videotimestamp": frame.get("Videotimestamp"),
                "match_clock_min": clock[0] if len(clock) > 0 else None,
                "match_clock_sec": clock[1] if len(clock) > 1 else None,
                "team_id": team_id,
                "player_id": player.get("id"),
                "player_x": player.get("x"),
                "player_y": player.get("y"),
                "player_visible": player.get("vis"),
            })
    if len(player_rows) >= 200:
        break

tracking_player_positions = pd.DataFrame(player_rows)
print(f"Sample player-position rows: {len(tracking_player_positions):,}")
display(tracking_player_positions.head(50))


Sample player-position rows: 220


,match_id,period,frame,Videotimestamp,match_clock_min,match_clock_sec,team_id,player_id,player_x,player_y,player_visible
0,679026,1,0,1.0,0,1,8561,1553280,48.88,61.48,False
1,679026,1,0,1.0,0,1,8561,1161475,54.64,9.92,False
2,679026,1,0,1.0,0,1,8561,1221904,55.40,27.14,False
3,679026,1,0,1.0,0,1,8561,1170336,66.40,38.27,False
4,679026,1,0,1.0,0,1,8561,1329572,52.08,21.28,False
5,679026,1,0,1.0,0,1,8561,1156913,52.78,53.32,False
6,679026,1,0,1.0,0,1,8561,1165515,64.50,18.98,False
7,679026,1,0,1.0,0,1,8561,1690083,52.44,47.52,False
8,679026,1,0,1.0,0,1,8561,1277032,53.27,33.66,False
9,679026,1,0,1.0,0,1,8561,1247852,91.82,33.72,False


In [22]:
# Camera/calibration points inside the nested `cam` column.
cam_rows = []
for frame in tracking_frames:
    cam = frame.get("cam")
    if not cam:
        continue
    clock = frame.get("match_clock") or [None, None]
    for point_index, point in enumerate(cam):
        cam_rows.append({
            "match_id": match_id,
            "period": frame.get("period"),
            "frame": frame.get("frame"),
            "Videotimestamp": frame.get("Videotimestamp"),
            "match_clock_min": clock[0] if len(clock) > 0 else None,
            "match_clock_sec": clock[1] if len(clock) > 1 else None,
            "cam_point_index": point_index,
            "cam_x": point[0] if isinstance(point, list) and len(point) > 0 else None,
            "cam_y": point[1] if isinstance(point, list) and len(point) > 1 else None,
        })
    if len(cam_rows) >= 120:
        break

tracking_cam_points = pd.DataFrame(cam_rows)
print(f"Sample camera-point rows: {len(tracking_cam_points):,}")
display(tracking_cam_points.head(50))


Sample camera-point rows: 120


,match_id,period,frame,Videotimestamp,match_clock_min,match_clock_sec,cam_point_index,cam_x,cam_y
0,679026,1,0,1.0,0,1,0,11.00,-4.44
1,679026,1,0,1.0,0,1,1,3059.81,-2795.66
2,679026,1,0,1.0,0,1,2,43.60,79.91
3,679026,1,0,1.0,0,1,3,76.11,80.24
4,679026,1,1,1.1,0,1,0,11.00,-4.59
5,679026,1,1,1.1,0,1,1,3042.07,-2788.70
6,679026,1,1,1.1,0,1,2,43.60,80.11
7,679026,1,1,1.1,0,1,3,76.25,80.49
8,679026,1,2,1.2,0,1,0,11.00,-4.74
9,679026,1,2,1.2,0,1,1,3024.32,-2781.74
